In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import random
import regex as re
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Load the data and create a vocabulary of characters and words
root = Path('..')
data_path = root / 'data' / 'shakespeare.txt'
text = open(data_path, 'r', encoding='utf-8').read()
print(f'Dataset length in characters: {len(text)}')

In [ ]:
tokens= text.encode('utf-8') # raw bytes
tokens = list(map(int, tokens)) # convert to a list of integers in range 0..255 for convenience

print(f'Text sample:\n {text[:100]}\n-----\nLength: {len(tokens)}\nTokens: {tokens}')

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
print(f'Stats: {stats}')

In [ ]:
top_pair = max(stats, key=stats.get)
print(f'Most common pair: {top_pair} with count {stats[top_pair]}')
print(f'"{chr(top_pair[0])}{chr(top_pair[1])}"')

In [ ]:
def merge(ids, pair, idx):
    # in the list of ints (ids), replace all consecutive occurences of pair with the new token idx
    new_ids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position and the pair matches, replace it
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

# print(merge([1, 2, 3, 1, 2, 3], (1, 2), 4))

tokens_replaced = merge(tokens, top_pair, 256)
print(f'Length original: {len(tokens)} | replaced: {len(tokens_replaced)}')

In [ ]:
VOCAB_SIZE = 276 # The desired size of the vocabulary
NUM_MERGES = VOCAB_SIZE - 256 # The number of merges to perform
ids = list(tokens) # copy the list of tokens, so we don't modify the original

merges = {}

for i in range(NUM_MERGES):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f'Merging pair {pair} into {idx}')
    ids = merge(ids, pair, idx)
    merges[pair] = idx

In [ ]:
print(f'Tokens length: {len(tokens)}')
print(f'Ids length: {len(ids)}')
print(f'Compression ratio: {len(tokens) / len(ids):.2f}')

In [ ]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
  # given ids (list of integers), return Python string
  tokens = b"".join(vocab[idx] for idx in ids)
  text = tokens.decode("utf-8", errors="replace")
  return text

print(decode([101]))

In [ ]:
def encode(text):
    # given a string / text, return list of integers (tokens)
    tokens = list(text.encode('utf-8'))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = max(stats, key=stats.get)
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode('Hello'))
print(decode(encode('Hello')))

test_text = decode(encode(text))
assert text == test_text, 'The implementation is incorrect'
print('The implementation is correct')

In [ ]:
# GPT-2 tokenization pattern with regex library
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello, world101! How      are you!?"))

In [ ]:
# minbpe exercise (GPT-4 tokenizer)

import tiktoken
enc = tiktoken.get_encoding("cl100k_base") # GPT-4 tokenizer
text = "Hello, world!"

print(enc.encode(text))
print(enc.decode(enc.encode(text)) == text)
